# svrecon benchmark

120 SVs (8 in-place types x 3 size classes x 5) simulated with insilicoSV, then scored by
svrecon under three CIGAR-validation modes, on a positive and a negative callset. Restricted to
in-place types (DEL, INV, DUP, DUP_INV, delINVdel, delINVdup, dupINVdel, dupINVdup) -- svrecon
does not yet support dispersed events (dDUP, nrTRA, rTRA, and their variants).

| arm | callset | assembly | expected |
|---|---|---|---|
| positive | seed 0 | seed 0 | 120/120 hits |
| negative | seed 1 | seed 0 | 0/120 hits |

Hits in the negative arm are false positives of that mode.

Configs live in `insilicoSV/` and `svrecon/` beside this notebook; it only runs them.

**Prerequisites:** run from this notebook's own directory (`workflows/`), with `insilicosv`
and `svrecon` pip-installed. All paths below are relative to it.

In [ ]:
%%bash
set -euo pipefail
for t in insilicosv svrecon; do command -v $t >/dev/null || { echo "$t not on PATH"; exit 1; }; done
test -f insilicoSV/release/insilicoSV.yaml || { echo 'run me from the workflows directory'; exit 1; }
echo ok

## 1. Download inputs

hg38 chr21 and its RepeatMasker intervals. The rmsk table is genome-wide (~180 MB),
piped down to chr21 rows. Skips anything already present.

In [ ]:
%%bash
set -euo pipefail
mkdir -p data

[ -f data/chr21.fa ] || \
  curl -fSL https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr21.fa.gz \
  | gunzip -c > data/chr21.fa

# rmsk.txt columns: 6 genoName, 7 genoStart, 8 genoEnd, 12 repClass
[ -f data/chr21.rmsk.bed ] || \
  curl -fSL https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/rmsk.txt.gz \
  | gunzip -c \
  | awk -F'\t' -v OFS='\t' '$6=="chr21" {print $6, $7, $8, $12}' \
  | sort -k1,1 -k2,2n > data/chr21.rmsk.bed

ls -lh data
wc -l < data/chr21.rmsk.bed

## 2. Simulate

Writes `sim.vcf` / `sim.hapA.fa` beside each config. ~1 min.

In [ ]:
%%bash
set -euo pipefail

# Writes sim.vcf / sim.hapA.fa beside each config; its reference path is cwd-relative.
(cd insilicoSV/release     && insilicosv -c insilicoSV.yaml)
(cd insilicoSV/release_neg && insilicosv -c insilicoSV.yaml)

## 3. Score

Six runs: 2 arms x 3 modes. Writes `svrecon.log` beside each config. ~10 min.

In [ ]:
%%bash
set -euo pipefail

(cd svrecon/release/similarity-only                && svrecon --config svrecon.yaml)
(cd svrecon/release/similarity-no-large-indels     && svrecon --config svrecon.yaml)
(cd svrecon/release/similarity-junction            && svrecon --config svrecon.yaml)
(cd svrecon/release_neg/similarity-only            && svrecon --config svrecon.yaml)
(cd svrecon/release_neg/similarity-no-large-indels && svrecon --config svrecon.yaml)
(cd svrecon/release_neg/similarity-junction        && svrecon --config svrecon.yaml)

### Results

In [ ]:
import json, re
from collections import Counter, defaultdict
from pathlib import Path

ARMS = ['release', 'release_neg']
MODES = ['similarity-only', 'similarity-no-large-indels', 'similarity-junction']
SIZES = ['small', 'medium', 'large']

def size_classes(vcf):
    """SVID -> size class, from its longest interval."""
    lens = defaultdict(list)
    for line in open(vcf):
        if line.startswith('#'):
            continue
        info = dict(kv.split('=', 1) for kv in line.split('\t')[7].split(';') if '=' in kv)
        lens[info['SVID']].append(int(info['SVLEN']))
    return {k: 'small' if max(v) < 500 else 'medium' if max(v) < 5000 else 'large'
            for k, v in lens.items()}

def outcomes(log):
    """SVID -> outcome, from the per-SV JSON each verbose log line ends with. Pretty-printed
    (indent=2) so it spans several lines; raw_decode anchored right at the '{' after the tab is
    indent-agnostic, unlike matching the whole blob with a single-line regex."""
    text = open(log).read()
    decoder = json.JSONDecoder()
    out = {}
    for m in re.finditer(r'\t\{\n', text):
        rec, _ = decoder.raw_decode(text, m.start() + 1)
        out[rec['svid']] = rec['outcome']
    return out

CLS = {a: size_classes(Path('insilicoSV') / a / 'sim.vcf') for a in ARMS}

rows = []
for arm in ARMS:
    for mode in MODES:
        out = outcomes(Path('svrecon') / arm / mode / 'svrecon.log')
        tot = Counter(CLS[arm][k] for k in out)
        hit = Counter(CLS[arm][k] for k, v in out.items() if v == 'hit')
        rows.append(['positive' if arm == 'release' else 'negative', mode,
                     *[f'{hit[s]}/{tot[s]}' for s in SIZES],
                     f'{sum(hit.values())}/{len(out)}',
                     f'{sum(hit.values()) / len(out):.2f}'])

hdr = ['arm', 'mode', *SIZES, 'total', 'precision']
w = [max(len(str(r[i])) for r in [hdr] + rows) for i in range(len(hdr))]
line = lambda r: '  '.join(str(c).ljust(w[i]) for i, c in enumerate(r))
print(line(hdr))
print('  '.join('-' * x for x in w))
for r in rows:
    print(line(r))
print('\nnegative arm: hits are false positives, lower is better')